# Data sources used in this study

This notebook provides a **minimal and reproducible record of the online data sources** used in the study.  
At this stage, the notebook only documents:

- the dataset names,
- the Google Earth Engine (GEE) dataset IDs,
- the official access channels,
- the study time range, and
- the study spatial scope.

No data processing is included in this notebook.


## Study scope

**Temporal scope used in this research**  
Hydrological years **2001–2024**

**Spatial scope used in this research**  
A user-defined **River corridor** shapefile

**Primary access platform**  
All datasets are accessed through **Google Earth Engine (GEE)**.


In [1]:
import pandas as pd

data_sources = [
    {
        "Dataset category": "Remote sensing",
        "Dataset name": "MODIS Terra Surface Reflectance Daily Global 1km and 500m",
        "GEE dataset ID": "MODIS/061/MOD09GA",
        "Platform / access channel": "Google Earth Engine Data Catalog",
        "Data provider": "NASA / USGS LP DAAC",
        "Variables used in this research": "MODIS surface reflectance imagery",
        "Study time range": "Hydrological years 2001–2024",
        "Study spatial scope": "River corridor shapefile"
    },
    {
        "Dataset category": "Remote sensing",
        "Dataset name": "MODIS Aqua Surface Reflectance Daily Global 1km and 500m",
        "GEE dataset ID": "MODIS/061/MYD09GA",
        "Platform / access channel": "Google Earth Engine Data Catalog",
        "Data provider": "NASA / USGS LP DAAC",
        "Variables used in this research": "MODIS surface reflectance imagery",
        "Study time range": "Hydrological years 2001–2024",
        "Study spatial scope": "River corridor shapefile"
    },
    {
        "Dataset category": "Reanalysis meteorology",
        "Dataset name": "ERA5-Land Daily Aggregated",
        "GEE dataset ID": "ECMWF/ERA5_LAND/DAILY_AGGR",
        "Platform / access channel": "Google Earth Engine Data Catalog",
        "Data provider": "ECMWF",
        "Variables used in this research": "2 m surface air temperature (SAT), net surface solar radiation, total precipitation",
        "Study time range": "Hydrological years 2001–2024",
        "Study spatial scope": "River corridor shapefile"
    }
]

df = pd.DataFrame(data_sources)
df

,Dataset category,Dataset name,GEE dataset ID,Platform / access channel,Data provider,Variables used in this research,Study time range,Study spatial scope
0,Remote sensing,MODIS Terra Surface Reflectance Daily Global 1...,MODIS/061/MOD09GA,Google Earth Engine Data Catalog,NASA / USGS LP DAAC,MODIS surface reflectance imagery,Hydrological years 2001–2024,River corridor shapefile
1,Remote sensing,MODIS Aqua Surface Reflectance Daily Global 1k...,MODIS/061/MYD09GA,Google Earth Engine Data Catalog,NASA / USGS LP DAAC,MODIS surface reflectance imagery,Hydrological years 2001–2024,River corridor shapefile
2,Reanalysis meteorology,ERA5-Land Daily Aggregated,ECMWF/ERA5_LAND/DAILY_AGGR,Google Earth Engine Data Catalog,ECMWF,"2 m surface air temperature (SAT), net surface...",Hydrological years 2001–2024,River corridor shapefile


## Official GEE access references

The datasets are accessed in Google Earth Engine using the following dataset IDs:

1. `MODIS/061/MOD09GA`
2. `MODIS/061/MYD09GA`
3. `ECMWF/ERA5_LAND/DAILY_AGGR`

These dataset IDs are sufficient for a reproducible description of the **online data source and access route**, because they directly identify the corresponding entries in the GEE Data Catalog.


In [2]:
gee_dataset_ids = [
    "MODIS/061/MOD09GA",
    "MODIS/061/MYD09GA",
    "ECMWF/ERA5_LAND/DAILY_AGGR"
]

for ds in gee_dataset_ids:
    print(ds)


MODIS/061/MOD09GA
MODIS/061/MYD09GA
ECMWF/ERA5_LAND/DAILY_AGGR


## Recommended wording for reproducible research

The study uses three online raster data sources that are accessed through Google Earth Engine (GEE):  
(1) **MODIS/061/MOD09GA**,  
(2) **MODIS/061/MYD09GA**, and  
(3) **ECMWF/ERA5_LAND/DAILY_AGGR**.  

The analysis period covers **hydrological years 2001–2024**, and the spatial extent is constrained by a user-defined **River corridor** shapefile.  
At this stage, this notebook documents only the **data source identifiers and access channels**; data extraction and processing are handled separately.


## Minimal examples for accessing and visualizing the data in GEE

The cells below add **simple access and visualization examples** without introducing any processing workflow.
They show how to:

1. initialize Google Earth Engine in Python,
2. read the user-defined **River corridor** shapefile,
3. visualize a **single-day MODIS NDSI** image, and
4. visualize a **single-day ERA5-Land 2 m air temperature** map.

> These are only demonstration examples for data access and display.  
> Cloud masking, quality filtering, temporal compositing, and other processing steps can be added in the next notebook/task.


In [3]:
# Optional: install packages if needed
# !pip install earthengine-api geemap geopandas


### 1. Initialize Earth Engine and load the River corridor shapefile

In [4]:
import ee
import geemap
import geopandas as gpd
from pathlib import Path

GEE_PROJECT = "ee-crisqiu"

# Authenticate only if needed
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

# Repository root
REPO_ROOT = Path.cwd().parent

# Shapefile path
SHAPEFILE_PATH = REPO_ROOT / "inputs" / "river_masks" / "Mackenzie_final.shp"

# Load river mask
river_gdf = gpd.read_file(SHAPEFILE_PATH)

# Convert to Earth Engine geometry
roi_fc = geemap.geopandas_to_ee(river_gdf)
roi = roi_fc.geometry()

print("River corridor loaded.")
print(river_gdf.head())

/srv/conda/envs/notebook/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


River corridor loaded.
   Id  ORIG_FID  InPoly_FID  SimPgnFlag  MaxSimpTol  MinSimpTol  \
0   0         0           0           0       200.0       200.0   

                                            geometry  
0  MULTIPOLYGON (((-2861312.144 1486148.284, -286...  


### 2. Visualize a single-day MODIS NDSI image

This example uses the MODIS surface reflectance product in GEE.  
For a simple NDSI demonstration, the code uses:

- `sur_refl_b04` as the green band
- `sur_refl_b06` as the SWIR band

You can switch between Terra and Aqua by changing `MODIS_DATASET_ID`:
- Terra: `MODIS/061/MOD09GA`
- Aqua: `MODIS/061/MYD09GA`


In [5]:
# Example date within the study period
DATE_STR = "2020-03-15"
MODIS_DATASET_ID = "MODIS/061/MOD09GA"   # change to MODIS/061/MYD09GA if needed

start = ee.Date(DATE_STR)
end = start.advance(1, "day")

# Multiple MODIS tiles may intersect the ROI on the same day,
# so mosaic() is used here for a simple visualization.
modis_image = (
    ee.ImageCollection(MODIS_DATASET_ID)
    .filterDate(start, end)
    .filterBounds(roi)
    .mosaic()
    .clip(roi)
)

ndsi = modis_image.normalizedDifference(["sur_refl_b04", "sur_refl_b06"]).rename("NDSI")

ndsi_vis = {
    "min": -1,
    "max": 1,
    "palette": ["1d1d1d", "6baed6", "bde5ff", "ffffff"]
}

Map = geemap.Map()
Map.centerObject(roi, 8)
Map.addLayer(ndsi, ndsi_vis, f"NDSI {DATE_STR}")
Map.addLayer(roi, {"color": "red"}, "River corridor")
Map


Map(center=[62.83447248204287, -125.3218744598663], controls=(WidgetControl(options=['position', 'transparent_…

### 3. Visualize a single-day ERA5-Land 2 m air temperature map

This example reads the `temperature_2m` band from `ECMWF/ERA5_LAND/DAILY_AGGR`.  
The original unit is **Kelvin**, so the example converts it to **°C** for display.


In [6]:
DATE_STR = "2020-03-15"

start = ee.Date(DATE_STR)
end = start.advance(1, "day")

era5_image = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .filterDate(start, end)
    .filterBounds(roi)
    .select("temperature_2m")
    .first()
    .clip(roi)
)

temp_c = era5_image.subtract(273.15).rename("temperature_2m_C")

temp_vis = {
    "min": -30,
    "max": 20,
    "palette": [
        "081d58", "225ea8", "1d91c0", "41b6c4",
        "7fcdbb", "c7e9b4", "ffffcc", "fdae61", "d7191c"
    ]
}

Map = geemap.Map()
Map.centerObject(roi, 8)
Map.addLayer(temp_c, temp_vis, f"2 m air temperature (°C) {DATE_STR}")
Map.addLayer(roi, {"color": "red"}, "River corridor")
Map


Map(center=[62.83447248204287, -125.3218744598663], controls=(WidgetControl(options=['position', 'transparent_…

## Notes

- The examples above focus only on **access and visualization**.
- For MODIS, the NDSI example is intentionally minimal and does **not** yet apply cloud masking or QA filtering.
- In the next task, these examples can be extended to include data quality screening, batch export, and reproducible preprocessing workflows.
